# LoRA (Low-Rank Adaptation)

**Tags:** parameter-efficient, finetuning
**Prerequisites:** Fine-tuning
**Related Concepts:** See flowchart below
**Source:** llm/concepts/lora.md

## TL;DR

Fine-tune LLMs by training only low-rank decomposition matrices, not all weights. Instead of updating W ∈ ℝ^(d × d), add A ∈ ℝ^(d × r) and B ∈ ℝ^(r × d) where r << d. Reduces parameters by 99%+, enables many task-specific models, minimal memory/compute cost.

## Core Intuition

Full fine-tuning updates all weights—expensive and redundant. The insight: weight changes during fine-tuning are low-rank. Store only the "directions of change" (r-rank factorization), not the full update. Recover full behavior with LoRA adapter on top of frozen base.

## How It Works

**Standard Fine-tuning (Baseline):**
```
Output = W · input
Update: W ← W - lr × ∇L
Cost: d × d parameters, gradient storage
```

**LoRA Approach:**
```
Output = W · input + (A · B^T) · input
        = W · input + Δ W · input

where:
  Δ W = A @ B  (low-rank decomposition)
  A ∈ ℝ^(d_in × r)  [trainable]
  B ∈ ℝ^(r × d_out) [trainable]
  r << min(d_in, d_out)  [rank, e.g., r=8]
```

**Typical rank reduction:**
- Full: 7B model ≈ 14B parameters to train
- LoRA (r=8): only 8 × d_hidden parameters per layer
- Result: 99% fewer parameters, ~90% training speedup

**Example: Adapting a Transformer:**
```
For each attention head and FFN layer:
  - Freeze: W (original weights)
  - Add: LoRA matrices A, B
  - Update: only A and B via backprop
  - Scaling: scale LoRA output by α/r to control magnitude
```

**Inference:**
```
Option 1: Keep base model + LoRA separate
  y = Wx + ABx  (merge at inference time)

Option 2: Merge weights offline
  W_LoRA = W + AB  (save merged checkpoint)
  y = W_LoRA × x  (same latency as full FT)
```

### Workflow Diagram**Note:** This is a basic workflow template. Review and customize based on specific concept.
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

## Key Properties & Trade-offs

/ Trade-offs

| Aspect | Full FT | LoRA | QLoRA |
|--------|---------|------|-------|
| Trainable params | 100% | 1-5% | 1-5% |
| Memory (training) | 100% | 10-20% | 2-4% |
| Training time | 100% | 30-50% faster | 50-70% faster |
| Accuracy | 100% | ~95-98% | ~95-98% |
| Inference overhead | None | Minimal (can merge) | Minimal |
| Deployment | 1 model | Multiple adapters | Multiple adapters |

**Rank (r) tradeoff:**
- r=4: fastest, lower quality (good for multiple fine-tunes)
- r=8: balanced (default)
- r=16: higher quality, slower (approaching full FT)
- r=64+: very close to full FT, marginal gains

**Scaling factor (α):**
- α controls magnitude of LoRA update relative to base model
- Common: α = 16 for r = 8 (scale = α/r)
- Too large: dominates base model; too small: negligible effect

### Code Implementation

```python
# Key imports for LoRA (Low-Rank Adaptation)
import numpy as np
import torch
from typing import Any

# LoRA (Low-Rank Adaptation) example implementation
class LoRA(LowRankAdaptation):
    """
    LoRA (Low-Rank Adaptation) implementation.
    This is a template - customize with actual code.
    """
    def __init__(self):
        pass

    def process(self, input_data: Any) -> Any:
        # Interview tip: Explain the core insight here
        return input_data
```

### Mathematical Formulation

Include LaTeX equations relevant to this concept.

**Example:**
$$\text{Output} = f(\text{Input})$$

## Related Concepts
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

### Common Interview Questions

**Q: What is LoRA (Low-Rank Adaptation) used for?**
A: [Add concise answer about practical application]

**Q: What are the main trade-offs of LoRA (Low-Rank Adaptation)?**
A: [Discuss pros/cons and when to use vs alternatives]

**Q: How does LoRA (Low-Rank Adaptation) compare to [related concept]?**
A: [Explain key differences and when to use each]

**Q: What are common mistakes when using LoRA (Low-Rank Adaptation)?**
A: [List 1-2 common pitfalls and how to avoid them]

**Q: Can you explain the intuition behind LoRA (Low-Rank Adaptation)?**
A: [Provide a simple analogy or explanation]

## References

- **Source Document:** `llm/concepts/lora.md`
- **Related Papers:** [Add relevant papers]
- **Implementations:**
  - HuggingFace: [Add links]
  - GitHub: [Add links]